In [0]:
import pyspark.sql 

In [0]:
spark_df = spark.read.table("workspace.gold.gold_stocks_prices")

risk_measure = spark.sql("SELECT Ticker, AVG(price_range) as price_range_avg FROM gold_stocks_prices GROUP BY Ticker ORDER BY price_range_avg DESC LIMIT 5")
# to include outliers use WHERE ABS(daily_return) > 3

flash_crashes = spark.sql("SELECT Ticker, Date, price_range, daily_return FROM gold_stocks_prices WHERE price_range > 10 ORDER BY price_range DESC")

cumulative_perf = spark.sql("SELECT Ticker, SUM(daily_return) as cumulative FROM gold_stocks_prices GROUP BY Ticker ORDER BY cumulative")

daily_performance = spark.sql("SELECT Date, Ticker, AVG(daily_return) as avg_daily_return FROM gold_stocks_prices GROUP BY Ticker, Date ORDER BY Date")

moving_average = spark.sql("SELECT Ticker, Date, Close, average_close, (CASE WHEN Close > average_close THEN 'Bullish' WHEN Close < average_close THEN 'Bearish' ELSE 'Wait' END) as trends FROM gold_stocks_prices")

risk_measure.write.format("delta").mode("overwrite").saveAsTable("workspace.platinum.risk_measure")
flash_crashes.write.format("delta").mode("overwrite").saveAsTable("workspace.platinum.flash_crashes")
cumulative_perf.write.format("delta").mode("overwrite").saveAsTable("workspace.platinum.cumulative_perf")
daily_performance.write.format("delta").mode("overwrite").saveAsTable("workspace.platinum.daily_performance")
moving_average.write.format("delta").mode("overwrite").saveAsTable("workspace.platinum.moving_average")